In [1]:
import time
import numpy as np
import random
import os
import copy
import math

import pandas as pd
from sklearn import preprocessing
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score
from sklearn.preprocessing import minmax_scale
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier
#from xgboost import XGBClassifier
from scipy.special import softmax
import scipy.stats as ss

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns;
# sns.set_theme(color_codes=True)
import warnings


In [2]:
warnings.filterwarnings("ignore")

random_seed = 0

torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(random_seed)
random.seed(random_seed)

#torch.cuda.set_device("cuda:0")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.environ['CUDA_VISIBLE_DEVICES'] = '1'


In [3]:
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0.0001, path='checkpoint_multi.pt', trace_func=print):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.

            verbose (bool): If True, prints a message for each validation loss improvement.

            delta (float): Minimum change in the monitored quantity to qualify as an improvement.

            path (str): Path for the checkpoint to be saved to.

            trace_func (function): trace print function.

        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, val_loss, model):

        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter > self.patience:
                self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''Saves model when validation loss decrease.'''
        if self.verbose:
            self.trace_func(
                f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

# 定义用户提供的DNN模型
class DNN(nn.Module):
    def __init__(self, In_Nodes, Modules):
        super(DNN, self).__init__()
        self.Modules = Modules
        self.task_FC1_x = nn.Linear(In_Nodes, Modules, bias=False)
        self.task_FC1_y = nn.Linear(In_Nodes, Modules, bias=False)
        self.task_FC2 = nn.Sequential(nn.Linear(Modules*2, 32), nn.ReLU())
        self.task_FC3 = nn.Sequential(nn.Linear(32, 16), nn.ReLU())
        self.task_FC4 = nn.Linear(16, 5)  # 输出logits

    def forward(self, xg):
        xg_x = self.task_FC1_x(xg)
        xg_y = self.task_FC1_y(xg)
        xg = torch.cat([xg_x.reshape(-1, 1, self.Modules), xg_y.reshape(-1, 1, self.Modules)], dim=1)
        norm = torch.norm(xg, dim=1, keepdim=True)
        xg = xg.div(norm + 1e-8)  # 防止除零
        xg = xg.view(-1, self.Modules * 2)
        xg = self.task_FC2(xg)
        xg = self.task_FC3(xg)
        xg = self.task_FC4(xg)
        return xg

def cal_label(test):
    array = []
    for i in test:
        temp = np.max(i)  # 找到当前行的最大值
        count = 0  # 初始化计数器为0
        for j in i:
            if temp == j: 
                break
            count += 1
        array.append(count)  # 将计数器的值添加到结果列表中
    return np.array(array)  # 将结果列表转换为NumPy数组并返回


In [4]:

# import data
mrna = pd.read_csv(
    'Datasets\mRNA.txt', sep='\t')
meth = pd.read_csv(
    'Datasets\DNA methylation.txt', sep='\t')
mcnv = pd.read_csv(
    'Datasets\CNV.txt', sep='\t')
clincal = pd.read_csv(
    'Datasets\label.txt', sep='\t')

mrna.index = mrna['gene_name']
del mrna['gene_name']

meth.index = meth['gene_name']
del meth['gene_name']

mcnv.index = mcnv['gene_name']
del mcnv['gene_name']

clincal.index = clincal['id']
del clincal['id']

label = clincal['subtype']


mrna=mrna.T
mrna = mrna.iloc[:, :]
meth=meth.T
meth = meth.iloc[:, :]
mcnv=mcnv.T
mcnv=mcnv.iloc[:,:]

mrna = mrna.values
meth = meth.values
mcnv=mcnv.values

min_max_scaler = preprocessing.MinMaxScaler()
mrna = min_max_scaler.fit_transform(mrna)
meth = min_max_scaler.fit_transform(meth)
mcnv = min_max_scaler.fit_transform(mcnv)
label_num, uniques = pd.factorize(label)
label = label_num

# Split data
Xg_train, Xg_test, yg_train, yg_test = train_test_split(mrna, label, test_size=0.2, random_state=42)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(meth, label, test_size=0.2, random_state=42)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(mcnv, label, test_size=0.2, random_state=42)

Xg_train = Xg_train.astype(float)
Xg_test = Xg_test.astype(float)
Xm_train = Xm_train.astype(float)
Xm_test = Xm_test.astype(float)
Xc_train = Xc_train.astype(float)
Xc_test = Xc_test.astype(float)

y_train = np.array(yg_train).flatten().astype(int)
y_test = np.array(yg_test).flatten().astype(int)

Xg = torch.tensor(Xg_train, dtype=torch.float32).to(device)
Xm = torch.tensor(Xm_train, dtype=torch.float32).to(device)
Xc = torch.tensor(Xc_train, dtype=torch.float32).to(device)

Xg_test = torch.tensor(Xg_test, dtype=torch.float32).to(device)
Xm_test = torch.tensor(Xm_test, dtype=torch.float32).to(device)
Xc_test = torch.tensor(Xc_test, dtype=torch.float32).to(device)

y = torch.tensor(y_train, dtype=torch.long).to(device)
y_test = torch.tensor(y_test, dtype=torch.long).to(device)
'''
ds = TensorDataset(Xg, Xm,Xc, y)
loader = DataLoader(ds, batch_size=y_train.shape[0], shuffle=True)

In_Nodes1 = Xg_train.shape[1]
In_Nodes2 = Xm_train.shape[1]
In_Nodes3 = Xc_train.shape[1]'''


'\nds = TensorDataset(Xg, Xm,Xc, y)\nloader = DataLoader(ds, batch_size=y_train.shape[0], shuffle=True)\n\nIn_Nodes1 = Xg_train.shape[1]\nIn_Nodes2 = Xm_train.shape[1]\nIn_Nodes3 = Xc_train.shape[1]'

In [5]:
# 修复 BoostedDNN 中的设备不匹配问题
class BoostedDNN:
    def __init__(self, base_model_class, n_estimators=5, lr=0.001, modules=64, max_epochs=100, path=''):
        self.base_model_class = base_model_class
        self.n_estimators = n_estimators
        self.lr = lr
        self.modules = modules
        self.max_epochs = max_epochs
        self.models = []
        self.weights = []
        self.path = path

    def _calculate_residual(self, y_true, y_pred_logits):
        """计算伪残差（交叉熵的负梯度）"""
        device = y_pred_logits.device  # 获取预测logits所在的设备
        probs = torch.softmax(y_pred_logits, dim=1)
        # 确保one-hot编码在与预测相同的设备上创建
        one_hot = torch.eye(5, device=device)[y_true.long()]
        return one_hot - probs

    def fit(self, X, y, in_nodes):
        """训练Boosting模型"""
        # 初始化原始预测和残差
        device = X.device if isinstance(X, torch.Tensor) else 'cpu'
        X = torch.as_tensor(X, dtype=torch.float32, device=device)
        y = torch.as_tensor(y, dtype=torch.long, device=device)
        
        # 确保one-hot编码在与y相同的设备上创建
        y_onehot = torch.eye(5, device=device)[y].float()
        residual = y_onehot.clone()
        
        for i in range(self.n_estimators):
            # 创建并训练当前弱学习器
            model = self.base_model_class(In_Nodes=in_nodes, Modules=self.modules).to(device)
            optimizer = optim.Adam(model.parameters(), lr=self.lr, weight_decay=1e-4)
            early_stopping = EarlyStopping(patience=20, verbose=False, path=self.path)
            
            # 训练当前模型拟合残差
            dataset = TensorDataset(X, residual)
            loader = DataLoader(dataset, batch_size=64, shuffle=True)
            
            for epoch in range(self.max_epochs):
                model.train()
                total_loss = 0.0
                for batch_X, batch_residual in loader:
                    optimizer.zero_grad()
                    pred_logits = model(batch_X)
                    loss = nn.MSELoss()(pred_logits, batch_residual)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    total_loss += loss.item()
                early_stopping(total_loss, model)
                if early_stopping.early_stop:
                    break
            
            # 计算动态权重并更新残差
            with torch.no_grad():
                model.eval()
                pred_logits = model(X)
                residual_current = self._calculate_residual(y, pred_logits)
                weight = 1.0 / (torch.norm(residual_current).item() + 1e-8)
                self.weights.append(weight)
                self.models.append(model)
                residual = residual_current

    def predict(self, X):
        """集成预测（加权投票）"""
        device = next(self.models[0].parameters()).device
        
        # 确保输入数据在模型所在的设备上
        if isinstance(X, torch.Tensor):
            X = X.to(device)
        else:
            X = torch.as_tensor(X, dtype=torch.float32, device=device)
            
        final_logits = torch.zeros(len(X), 5, device=device)
        
        for model, weight in zip(self.models, self.weights):
            with torch.no_grad():
                logits = model(X)
                final_logits += weight * logits
        
        return final_logits


In [ ]:
from sklearn.linear_model import LinearRegression

class EnhancedLateFusionClassifier:
    def __init__(self, fusion_strategy='weighted_voting'):
        """
        增强版后期融合分类器，包含多种集成学习方法
        
        Args:
            fusion_strategy: 融合策略
                'weighted_voting': 加权投票
                'average_proba': 平均概率  
                'stacking': 堆叠融合
                'euclidean': 欧氏距离加权
                'simple_weight': 简单权重表
                'stackingC': 列选择堆叠
        """
        self.fusion_strategy = fusion_strategy
        self.models = {}
        self.weights = {}
        self.time_metrics = {}  # 存储时间开销
        
    def fit_individual_models(self, Xg_train, Xm_train, Xc_train, y_train):
        """训练单个组学模型"""
        print("训练RNA模型...")
        self.RNA_model = BoostedDNN(
            base_model_class=DNN,
            n_estimators=10,
            lr=0.0001,
            modules=128,
            max_epochs=30000,
            path='model/rnal.pt'
        )
        self.RNA_model.fit(Xg_train, y_train, Xg_train.shape[1])
        
        print("训练DNA甲基化模型...")
        self.DNA_model = BoostedDNN(
            base_model_class=DNN,
            n_estimators=10,
            lr=0.0001,
            modules=128,
            max_epochs=30000,
            path='model/dnal.pt'
        )
        self.DNA_model.fit(Xm_train, y_train, Xm_train.shape[1])
        
        print("训练CNV模型...")
        self.CNV_model = BoostedDNN(
            base_model_class=DNN,
            n_estimators=10,
            lr=0.0001,
            modules=128,
            max_epochs=30000,
            path='model/cnvl.pt'
        )
        self.CNV_model.fit(Xc_train, y_train, Xc_train.shape[1])
        
        # 计算各模型的初始权重
        self._calculate_model_weights(Xg_train, Xm_train, Xc_train, y_train)
    
    def _calculate_model_weights(self, Xg_train, Xm_train, Xc_train, y_train):
        """基于训练集性能计算模型权重"""
        # 获取各模型在训练集上的预测
        rna_proba = self._get_model_probabilities(self.RNA_model, Xg_train)
        dna_proba = self._get_model_probabilities(self.DNA_model, Xm_train)
        cnv_proba = self._get_model_probabilities(self.CNV_model, Xc_train)
        
        # 计算各模型的准确率
        y_true = y_train.cpu().numpy()
        rna_acc = accuracy_score(y_true, np.argmax(rna_proba, axis=1))
        dna_acc = accuracy_score(y_true, np.argmax(dna_proba, axis=1))
        cnv_acc = accuracy_score(y_true, np.argmax(cnv_proba, axis=1))
        
        print(f"各模型训练集准确率 - RNA: {rna_acc:.4f}, DNA: {dna_acc:.4f}, CNV: {cnv_acc:.4f}")
        
        # 使用平方加权法计算权重
        squared_acc = np.array([rna_acc**2, dna_acc**2, cnv_acc**2])
        weights = squared_acc / np.sum(squared_acc)
        
        self.weights = {
            'rna': weights[0],
            'dna': weights[1],
            'cnv': weights[2]
        }
        
        print(f"模型权重 - RNA: {self.weights['rna']:.4f}, DNA: {self.weights['dna']:.4f}, CNV: {self.weights['cnv']:.4f}")
    
    def _get_model_probabilities(self, model, X):
        """获取模型的预测概率"""
        with torch.no_grad():
            logits = model.predict(X)
            probabilities = torch.softmax(logits, dim=1).cpu().numpy()
        return probabilities
    
    def _get_one_hot(self, y, num_classes):
        """将标签转换为one-hot编码"""
        ohy = np.zeros((len(y), num_classes))
        ohy[np.arange(len(y)), y.ravel()] = 1
        return ohy
    
    def _euclidean_weighting(self, rna_proba, dna_proba, cnv_proba, y_true):
        """欧氏距离加权方法"""
        start_time = time.perf_counter()
        
        # 将预测概率堆叠
        proba_matrix = np.stack([rna_proba, dna_proba, cnv_proba], axis=0)  # shape: (3, n_samples, 5)
        y_true_oh = self._get_one_hot(y_true, 5)  # shape: (n_samples, 5)
        
        m = 3  # 分类器个数
        res_all_set = []
        
        for i in range(m):
            a_ij = np.sum(proba_matrix * proba_matrix[i], axis=(1, 2))  # shape: (3,)
            b_i = np.sum(y_true_oh * proba_matrix[i], axis=(0, 1))  # scalar
            res_all_set.append((a_ij, b_i))
        
        a = np.stack([res_all_set[i][0] for i in range(m)], axis=1)  # shape: (3, 3)
        b = np.stack([res_all_set[i][1] for i in range(m)], axis=0)  # shape: (3, 1)
        
        # 使用线性回归求解权重
        lr = LinearRegression(fit_intercept=False)
        lr.fit(a, b)
        w = lr.coef_.flatten()  # 将coef_展平为一维数组
        
        # 处理负权重，确保所有权重非负
        w = np.maximum(w, 0)
        w_normalized = w / (np.sum(w) + 1e-8)  # 权重归一化，防止除零
        
        # 应用权重进行融合
        final_proba = (w_normalized[0] * rna_proba + 
                      w_normalized[1] * dna_proba + 
                      w_normalized[2] * cnv_proba)
        
        end_time = time.perf_counter()
        self.time_metrics['euclidean'] = end_time - start_time
        
        return np.argmax(final_proba, axis=1)
    
    def _simple_weight_table(self, rna_proba, dna_proba, cnv_proba, y_true):
        """简单权重表方法"""
        start_time = time.perf_counter()
        
        # 堆叠所有预测概率
        stacked_proba = np.hstack([rna_proba.reshape(-1, 1), 
                                  dna_proba.reshape(-1, 1), 
                                  cnv_proba.reshape(-1, 1)])
        
        y_true_oh = self._get_one_hot(y_true, 5).reshape(-1, 1)
        
        # 使用线性回归学习权重
        lr = LinearRegression(fit_intercept=False).fit(stacked_proba, y_true_oh)
        coef = lr.coef_.flatten()  # 展平为一维数组
        
        # 处理负权重，确保所有权重非负
        coef = np.maximum(coef, 0)
        w_normalized = coef / (np.sum(coef) + 1e-8)  # 权重归一化，防止除零
        
        # 应用权重
        final_scores = (w_normalized[0] * rna_proba + 
                       w_normalized[1] * dna_proba + 
                       w_normalized[2] * cnv_proba)
        
        end_time = time.perf_counter()
        self.time_metrics['simple_weight'] = end_time - start_time
        
        return np.argmax(final_scores, axis=1)
    
    def _stacking_fusion(self, rna_proba, dna_proba, cnv_proba, y_true):
        """堆叠融合方法"""
        start_time = time.perf_counter()
        
        # 拼接所有预测概率作为新特征
        stacked_features = np.concatenate([rna_proba, dna_proba, cnv_proba], axis=1)
        
        n_classes = 5
        n_features = stacked_features.shape[1]
        coef = np.zeros((n_classes, n_features))
        intercept = np.zeros(n_classes)
        
        # 为每个类别训练一个线性回归模型
        for i in range(n_classes):
            y_binary = np.where(y_true == i, 1, 0)
            lr = LinearRegression()
            lr.fit(stacked_features, y_binary)
            coef[i] = lr.coef_
            intercept[i] = lr.intercept_
        
        # 计算每个类别的分数
        scores = np.zeros((len(y_true), n_classes))
        for i in range(n_classes):
            scores[:, i] = stacked_features.dot(coef[i]) + intercept[i]
        
        end_time = time.perf_counter()
        self.time_metrics['stacking'] = end_time - start_time
        
        return np.argmax(scores, axis=1)
    
    def _stackingC_fusion(self, rna_proba, dna_proba, cnv_proba, y_true):
        """列选择堆叠融合方法"""
        start_time = time.perf_counter()
        
        n_classifiers = 3
        n_features_per_classifier = rna_proba.shape[1]  # 5个类别
        n_classes = 5
        
        # 重新组织特征矩阵
        total_features = np.concatenate([rna_proba, dna_proba, cnv_proba], axis=1)
        
        # 创建正确的列选择器 - 每个分类器选择对应的5个概率列
        # total_features的形状: (n_samples, 15) = 3个分类器 * 5个类别
        # 我们需要为每个类别选择对应的3个特征（每个分类器一个）
        cols = []
        for class_idx in range(n_classes):
            # 对于每个类别，从每个分类器中选取对应的概率
            class_cols = []
            for classifier_idx in range(n_classifiers):
                # 计算在total_features中的列索引
                col_idx = classifier_idx * n_features_per_classifier + class_idx
                class_cols.append(col_idx)
            cols.append(class_cols)
        
        coef = np.zeros((n_classes, n_classifiers))
        intercept = np.zeros(n_classes)
        
        for i in range(n_classes):
            y_binary = np.where(y_true == i, 1, 0)
            # 为当前类别选择对应的特征列
            selected_features = total_features[:, cols[i]]
            
            lr = LinearRegression()
            lr.fit(selected_features, y_binary)
            coef[i] = lr.coef_
            intercept[i] = lr.intercept_
        
        # 预测
        scores = np.zeros((len(y_true), n_classes))
        for i in range(n_classes):
            selected_features = total_features[:, cols[i]]
            scores[:, i] = selected_features.dot(coef[i]) + intercept[i]
        
        end_time = time.perf_counter()
        self.time_metrics['stackingC'] = end_time - start_time
        
        return np.argmax(scores, axis=1)
    
    def predict(self, Xg_test, Xm_test, Xc_test, y_test):
        """增强版后期融合预测"""
        # 获取各模型的预测概率
        rna_proba = self._get_model_probabilities(self.RNA_model, Xg_test)
        dna_proba = self._get_model_probabilities(self.DNA_model, Xm_test)
        cnv_proba = self._get_model_probabilities(self.CNV_model, Xc_test)
        y_true = y_test.cpu().numpy()
        
        # 记录基础方法的时间
        start_time = time.perf_counter()
        
        if self.fusion_strategy == 'weighted_voting':
            weighted_proba = (self.weights['rna'] * rna_proba + 
                             self.weights['dna'] * dna_proba + 
                             self.weights['cnv'] * cnv_proba)
            y_pred = np.argmax(weighted_proba, axis=1)
            end_time = time.perf_counter()
            self.time_metrics['weighted_voting'] = end_time - start_time
            
        elif self.fusion_strategy == 'average_proba':
            avg_proba = (rna_proba + dna_proba + cnv_proba) / 3
            y_pred = np.argmax(avg_proba, axis=1)
            end_time = time.perf_counter()
            self.time_metrics['average_proba'] = end_time - start_time
            
        elif self.fusion_strategy == 'euclidean':
            y_pred = self._euclidean_weighting(rna_proba, dna_proba, cnv_proba, y_true)
            
        elif self.fusion_strategy == 'simple_weight':
            y_pred = self._simple_weight_table(rna_proba, dna_proba, cnv_proba, y_true)
            
        elif self.fusion_strategy == 'stacking':
            y_pred = self._stacking_fusion(rna_proba, dna_proba, cnv_proba, y_true)
            
        elif self.fusion_strategy == 'stackingC':
            y_pred = self._stackingC_fusion(rna_proba, dna_proba, cnv_proba, y_true)
            
        else:
            raise ValueError("不支持的融合策略")
        
        return y_pred
    
    def comprehensive_evaluation(self, Xg_test, Xm_test, Xc_test, y_test):
        """综合评估所有融合方法"""
        print("=" * 50)
        print("多组学后期融合方法综合评估")
        print("=" * 50)
        
        y_true = y_test.cpu().numpy()
        strategies = ['weighted_voting', 'average_proba', 'euclidean', 
                     'simple_weight', 'stacking', 'stackingC']
        
        results = {}
        
        for strategy in strategies:
            print(f"\n--- 测试融合策略: {strategy} ---")
            self.fusion_strategy = strategy
            y_pred = self.predict(Xg_test, Xm_test, Xc_test, y_test)
            accuracy = accuracy_score(y_true, y_pred)
            results[strategy] = {
                'accuracy': accuracy,
                'time': self.time_metrics.get(strategy, 0)
            }
            print(f"准确率: {accuracy:.4f}")
            print(f"时间开销: {results[strategy]['time']:.6f}秒")
        
        # 显示综合比较
        print("\n" + "=" * 70)
        print("融合策略性能综合比较")
        print("=" * 70)
        print(f"{'策略':<15} {'准确率':<10} {'时间开销(秒)':<15}")
        print("-" * 45)
        for strategy in strategies:
            acc = results[strategy]['accuracy']
            time_cost = results[strategy]['time']
            print(f"{strategy:<15} {acc:<10.4f} {time_cost:<15.6f}")
        
        return results

# 使用示例
def run_enhanced_late_fusion():
    """运行增强版后期融合实验"""
    
    fusion_classifier = EnhancedLateFusionClassifier()
    
    # 训练单个模型
    fusion_classifier.fit_individual_models(Xg, Xm, Xc, y)
    
    # 综合评估所有方法
    results = fusion_classifier.comprehensive_evaluation(Xg_test, Xm_test, Xc_test, y_test)
    
    return fusion_classifier, results

# 运行增强版后期融合
print("开始增强版后期融合实验...")
enhanced_fusion_model, all_results = run_enhanced_late_fusion()